# Home Credit Default Risk Competition
In this notebook, we will do feature engineering on the whole dataset, then we will select the most pertinent features to be used in training our ML models.
A reminder, The objective of this competition is to use historical loan application data to predict whether or not an applicant will be able to repay a loan. This is a standard supervised classification task:

* __Supervised__: The labels are included in the training data and the goal is to train a model to learn to predict the labels from the features
* __Classification__: The label is a binary variable, 0 (will repay loan on time), 1 (will have difficulty repaying loan)

__This notebook inspired by the following kernels:__
* [LightGBM with Simple Features](https://www.kaggle.com/code/jsaguiar/lightgbm-with-simple-features/scriptd)
* [Start Here: A Gentle Introduction](https://www.kaggle.com/code/willkoehrsen/start-here-a-gentle-introduction)

This notebook organized as follow:
 * Load and preprocess train & test dataset
 * Join others datasets into train & test, after adding new features
 * Delete columns with missing data more than 75%
 * Delete columns with correlated score more than 80%
 * select 107 most correlated columns (features) with the target as the main features to be used in training ML models

## 1. Imports

In [20]:
import sys
import os
import numpy as np
import pandas as pd
import gc
import time
from contextlib import contextmanager
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import sys
sys.path.append('../')
from src.functions import *

## 2. Functions definitions

In [2]:
@contextmanager
def timer(title):
    t0 = time.time()
    yield
    print("{} - done in {:.0f}s".format(title, time.time() - t0))

In [3]:
def remove_missing_columns(data, threshold = 90):
    """
    This function calculate the percentage of missing data in each column
    Then delete the columns which greater than defined threshold
    """
    # Calculate missing stats for whole dataset
    data_miss = pd.DataFrame(data.isnull().sum())
    data_miss['percent'] = 100 * data_miss[0] / len(data)
    
    # list of missing columns for train and test
    missing_columns = list(data_miss.index[data_miss['percent'] > threshold])

    # Print information
    print('There are %d columns with greater than %d%% missing values.' % (len(missing_columns), threshold))
    
    # Drop the missing columns and return
    data = data.drop(columns = missing_columns)
    
    return data

## 3. Feature engineering

### 1. Read & preprocess the dataset (train&test)
Put train & test in the same dataset to faciliate preprocessing, where:
* Factorizing the categorical features with Binary encode
* Onehot encoding applied on categorical features with more than two values
* Remove outliers for the feature DAYS_EMPLOYED

In [4]:
#num_rows = 15000
df = application_train_test()

Train samples: 307511, test samples: 48744
Original Memory Usage: 0.38 gb.
New Memory Usage: 0.21 gb.


In [5]:
df.shape

(356251, 246)

### 2. Merge the tables (bureau & bureau balance) with the dataset (train&test)

In [6]:
    with timer("Process bureau and bureau_balance"):
        bureau = bureau_and_balance()
        # print("Bureau df shape:", bureau.shape)
        df = df.join(bureau, how='left', on='SK_ID_CURR')
        del bureau
        gc.collect()

bureau shape  (1716428, 40)
bureau_blance shape  (27299925, 11)
Original Memory Usage: 0.29 gb.
New Memory Usage: 0.14 gb.
Process bureau and bureau_balance - done in 23s


In [7]:
df.shape

(356251, 362)

### 3. Merge the table (Process previous applications) with the dataset (train&test)

In [8]:
    with timer("Process previous_applications"):
        prev = previous_applications()
        print("Previous applications shape after preprocessing :", prev.shape)
        df = df.join(prev, how='left', on='SK_ID_CURR')
        del prev
        gc.collect()

Original Memory Usage: 0.68 gb.
New Memory Usage: 0.34 gb.
Previous applications shape after preprocessing : (338857, 249)
Process previous_applications - done in 33s


In [9]:
df.shape

(356251, 611)

### 4. Merge the table (POS-CASH balance) with the dataset (train&test)

In [10]:
    with timer("Process POS-CASH balance"):
        pos = pos_cash()
        print("Pos-cash balance shape after preprocessing :", pos.shape)
        df = df.join(pos, how='left', on='SK_ID_CURR')
        del pos
        gc.collect()

POS_CASH_balance shape before analysing  (10001358, 8)
Original Memory Usage: 0.05 gb.
New Memory Usage: 0.03 gb.
Pos-cash balance shape after preprocessing : (337252, 18)
Process POS-CASH balance - done in 16s


In [11]:
df.shape

(356251, 629)

### 5. Merge the table (installments payments) with the dataset (train&test)

In [12]:
    with timer("Process installments payments"):
        ins = installments_payments()
        print("Installments payments shape after preprocessing :", ins.shape)
        df = df.join(ins, how='left', on='SK_ID_CURR')
        del ins
        gc.collect()

Original Memory Usage: 0.07 gb.
New Memory Usage: 0.04 gb.
Installments payments shape after preprocessing : (339587, 26)
Process installments payments - done in 27s


In [13]:
df.shape

(356251, 655)

### 6. Merge the table (credit card balance) with the dataset (train&test)

In [14]:
    with timer("Process credit card balance"):
        cc = credit_card_balance()
        print("Credit card balance shape after preprocessing :", cc.shape)
        df = df.join(cc, how='left', on='SK_ID_CURR')
        del cc
        gc.collect()

Original Memory Usage: 0.11 gb.
New Memory Usage: 0.05 gb.
Credit card balance shape after preprocessing : (103558, 141)
Process credit card balance - done in 19s


In [15]:
df.shape

(356251, 796)

In [16]:
 round(df.memory_usage().sum() / 1e9, 2)

np.float64(1.07)

### 7.Save the dataset as a parquet

In [21]:
# checking if the directory exist or not.
if not os.path.exists("../../data/preprocessed_data"):
    # if the directory is not present then create it.
    os.makedirs("../../data/preprocessed_data")

In [22]:
df.to_parquet("../../data/preprocessed_data/all_data.parquet",index=False)

In [ ]:
# df2= pd.read_parquet("../data/all_data.parquet")

## 4. Feature selection

### 1. Checking for missing data
**Delete columns with  missing data more than 75%**

In [23]:
total = df.isnull().sum().sort_values(ascending = False)
percent = (df.isnull().sum()/df.isnull().count()*100).sort_values(ascending = False)
missing_data  = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])

In [24]:
missing_data.head(50)

,Total,Percent
REFUSED_AMT_DOWN_PAYMENT_MIN,303648,85.234287
REFUSED_AMT_DOWN_PAYMENT_MAX,303648,85.234287
REFUSED_RATE_DOWN_PAYMENT_MAX,303648,85.234287
REFUSED_RATE_DOWN_PAYMENT_MEAN,303648,85.234287
REFUSED_AMT_DOWN_PAYMENT_MEAN,303648,85.234287
REFUSED_RATE_DOWN_PAYMENT_MIN,303648,85.234287
REFUSED_APP_CREDIT_PERC_VAR,298034,83.658432
CC_AMT_PAYMENT_CURRENT_VAR,284649,79.901249
CC_AMT_DRAWINGS_OTHER_CURRENT_VAR,284559,79.875986
CC_CNT_DRAWINGS_ATM_CURRENT_VAR,284559,79.875986


In [25]:
df2 = remove_missing_columns(df,threshold =75)

There are 35 columns with greater than 75% missing values.


In [26]:
df2.shape

(356251, 761)

In [27]:
del df
gc.collect()

11556

### 2. Correlation

To get more reasonable analysis we will do the correlation on the train dataset only

In [28]:
# Divide in training/validation and test data
train_df = df2[df2['TARGET'].notnull()]
test_df = df2[df2['TARGET'].isnull()]

In [29]:
train_df.shape

(307507, 761)

In [30]:
test_df.shape

(48744, 761)

In [31]:
del df2
gc.collect()

0

In [32]:
# Calculate all correlations in train dataframe
corrs = train_df.corr()

#### 1. Correlation with the target

In [33]:
corrs = corrs.sort_values('TARGET', ascending = False)

# Ten most positive correlations
pd.DataFrame(corrs['TARGET'].head(10))

,TARGET
TARGET,1.000000
CC_CNT_DRAWINGS_CURRENT_MAX,0.101389
BURO_DAYS_CREDIT_MEAN,0.089731
CC_AMT_BALANCE_MEAN,0.087177
CC_AMT_TOTAL_RECEIVABLE_MEAN,0.086490
CC_AMT_RECIVABLE_MEAN,0.086478
CC_AMT_RECEIVABLE_PRINCIPAL_MEAN,0.086062
CC_CNT_DRAWINGS_CURRENT_MEAN,0.082520
DAYS_BIRTH,0.078242
PREV_NAME_CONTRACT_STATUS_Refused_MEAN,0.077681


In [34]:
# Ten most negative correlations
pd.DataFrame(corrs['TARGET'].dropna().tail(10))

,TARGET
CC_COUNT,-0.060481
PREV_NAME_CONTRACT_STATUS_Approved_MEAN,-0.063526
ACTIVE_MONTHS_BALANCE_SIZE_MEAN,-0.065154
DAYS_EMPLOYED_PERC,-0.067952
PREV_CODE_REJECT_REASON_XAP_MEAN,-0.073938
BURO_CREDIT_ACTIVE_Closed_MEAN,-0.079369
BURO_MONTHS_BALANCE_SIZE_MEAN,-0.080193
EXT_SOURCE_1,-0.155317
EXT_SOURCE_2,-0.160471
EXT_SOURCE_3,-0.178926


#### 2. Correlation between features

Collinear Variables
We can calculate not only the correlations of the variables with the target, but also the correlation of each variable with every other variable. This will allow us to see if there are highly collinear variables that should perhaps be removed from the data.

In [35]:
# Set the threshold
threshold = 0.8

# Empty dictionary to hold correlated variables
above_threshold_vars = {}

# For each column, record the variables that are above the threshold
for col in corrs:
    above_threshold_vars[col] = list(corrs.index[corrs[col] > threshold])

For each of these pairs of highly correlated variables, we only want to remove one of the variables. The following code creates a set of variables to remove by only adding one of each pair.

In [36]:
# Track columns to remove and columns already examined
cols_to_remove = []
cols_seen = []
cols_to_remove_pair = []

# Iterate through columns and correlated columns
for key, value in above_threshold_vars.items():
    # Keep track of columns already examined
    cols_seen.append(key)
    for x in value:
        if x == key:
            next
        else:
            # Only want to remove one in a pair
            if x not in cols_seen:
                cols_to_remove.append(x)
                cols_to_remove_pair.append(key)
            
cols_to_remove = list(set(cols_to_remove))
print('Number of columns to remove: ', len(cols_to_remove))

Number of columns to remove:  221


***As we need the ids of the applications, for the reason of  tracabilty, we will keep the column 'SK_ID_CURR'***

In [37]:
cols_to_remove.remove('SK_ID_CURR')

#### 3. Delete correlated features from train & test dataset

In [38]:
train_corrs_removed = train_df.drop(columns = cols_to_remove)

Delete the same columns from test dataset

In [39]:
train_corrs_removed.shape

(307507, 541)

In [40]:
# train_corrs_removed.columns.values

In [41]:
test_corrs_removed = test_df.drop(columns = cols_to_remove)

In [42]:
test_corrs_removed.shape

(48744, 541)

In [43]:
test_corrs_removed.drop(['TARGET'],axis=1,inplace=True)

In [44]:
test_corrs_removed.shape

(48744, 540)

**Save dataset into parquets**

In [45]:
train_corrs_removed.to_parquet("../../data/preprocessed_data/train_withoutCorr.parquet",index=False)

In [46]:
test_corrs_removed.to_parquet("../../data/preprocessed_data/test_withoutCorr.parquet",index=False)

In [47]:
del train_df
gc.collect()

0

In [48]:
del test_df
gc.collect()

0

#### 4. Select the most correlated features with the target

For the reason of simplicity, we will choose the first 100 features and last 100 features from the table of correlation with the target

In [49]:
target_corr_df = pd.DataFrame(corrs['TARGET'].dropna())

In [50]:
feature_most_corr_pos = target_corr_df.index.values[1:101]
print(len(feature_most_corr_pos))

100


In [51]:
feature_most_corr_neg = target_corr_df.index.values[-100:]
print(len(feature_most_corr_neg))

100


In [52]:
feature_most_corr_list = np.union1d(feature_most_corr_pos,feature_most_corr_neg).tolist()
len(feature_most_corr_list)

200

In [53]:
#feature_most_corr_list = feature_most_corr.tolist()

In [54]:
#len(feature_most_corr_list

In [55]:
feature_most_corr_list.extend(['TARGET', 'SK_ID_CURR'])
min_train = train_corrs_removed.loc[:, train_corrs_removed.columns.isin(feature_most_corr_list)] 
min_train.shape

(307507, 109)

In [56]:
min_test = test_corrs_removed.loc[:, test_corrs_removed.columns.isin(feature_most_corr_list)] 
min_test.shape

(48744, 108)

***As we can see, there are alot of features, have a strong correlation with the target, have been deleted because of strong correlation between them. So we will keep in total 107 features to be used in training process***

#### 5. Save the min dataset into parquet

In [57]:
min_train.to_parquet("../../data/preprocessed_data/min_train_108f.parquet",index=False)
min_test.to_parquet("../../data/preprocessed_data/min_test_108f.parquet",index=False)